In [75]:
import pandas as pd

In [76]:
df = pd.read_csv("international_match_results.csv")

 
df.head()


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [77]:
df.shape

(49477, 9)

In [78]:
df.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49472,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True
49473,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49474,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49475,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49476,2026-06-27,Croatia,Ghana,NaN,NaN,FIFA World Cup,Philadelphia,United States,True


In [79]:
df.isna().sum()

date           0
home_team      0
away_team      0
home_score    68
away_score    68
tournament     0
city           0
country        0
neutral        0
dtype: int64

In [80]:
df.tail(72)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49405,2026-06-11,Mexico,South Africa,2.0,0.0,FIFA World Cup,Mexico City,Mexico,False
49406,2026-06-11,South Korea,Czech Republic,2.0,1.0,FIFA World Cup,Zapopan,Mexico,True
49407,2026-06-12,Canada,Bosnia and Herzegovina,1.0,1.0,FIFA World Cup,Toronto,Canada,False
49408,2026-06-12,United States,Paraguay,4.0,1.0,FIFA World Cup,Inglewood,United States,False
49409,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True
...,...,...,...,...,...,...,...,...,...
49472,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True
49473,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49474,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49475,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True


In [83]:
elo_ratings = pd.read_csv("eloratings.csv")

df["date"] = pd.to_datetime(df["date"])
elo_ratings["date"] = pd.to_datetime(elo_ratings["date"], format="mixed")

world_cup_cutoff = pd.Timestamp("2026-06-10")
world_cup_fixtures = df[df["date"] > world_cup_cutoff].copy()
historical_matches = df[df["date"] <= world_cup_cutoff].copy()


df = historical_matches.sort_values("date").reset_index(drop=True)
elo_ratings = elo_ratings.sort_values("date").reset_index(drop=True)

world_cup_fixtures.head()


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49405,2026-06-11,Mexico,South Africa,2.0,0.0,FIFA World Cup,Mexico City,Mexico,False
49406,2026-06-11,South Korea,Czech Republic,2.0,1.0,FIFA World Cup,Zapopan,Mexico,True
49407,2026-06-12,Canada,Bosnia and Herzegovina,1.0,1.0,FIFA World Cup,Toronto,Canada,False
49408,2026-06-12,United States,Paraguay,4.0,1.0,FIFA World Cup,Inglewood,United States,False
49409,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True


In [84]:
import numpy as np

world_cup_fixtures.head()



,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49405,2026-06-11,Mexico,South Africa,2.0,0.0,FIFA World Cup,Mexico City,Mexico,False
49406,2026-06-11,South Korea,Czech Republic,2.0,1.0,FIFA World Cup,Zapopan,Mexico,True
49407,2026-06-12,Canada,Bosnia and Herzegovina,1.0,1.0,FIFA World Cup,Toronto,Canada,False
49408,2026-06-12,United States,Paraguay,4.0,1.0,FIFA World Cup,Inglewood,United States,False
49409,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True


In [85]:
import numpy as np

world_cup_fixtures[["home_score", "away_score"]] = np.nan

In [86]:
world_cup_fixtures.to_csv("world_cup_fixtures.csv", index=False)

In [87]:
elo_ratings.head()


,date,team,rating,change
0,1872-11-30,England,2003.0,3
1,1872-11-30,Scotland,1997.0,-3
2,1873-03-08,England,2014.0,11
3,1873-03-08,Scotland,1986.0,-11
4,1874-03-07,England,2006.0,-8


In [88]:
 
df.tail()


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49400,2026-06-09,Ethiopia,Malawi,1.0,1.0,Friendly,Dire Dawa,Ethiopia,False
49401,2026-06-10,England,Costa Rica,3.0,0.0,Friendly,Orlando,United States,True
49402,2026-06-10,Portugal,Nigeria,2.0,1.0,Friendly,Leiria,Portugal,False
49403,2026-06-10,Bolivia,Algeria,0.0,4.0,Friendly,Kansas City,United States,True
49404,2026-06-10,Afghanistan,Pakistan,0.0,2.0,Diamond Jubilee International Football Tournament,Malé,Maldives,True


In [89]:
world_cup_fixtures.isna().sum()

date           0
home_team      0
away_team      0
home_score    72
away_score    72
tournament     0
city           0
country        0
neutral        0
dtype: int64

In [90]:

elo_lookup = elo_ratings[["date", "team", "rating"]].copy()

home_elo = pd.merge_asof(
    df.sort_values("date"),
    elo_lookup.rename(columns={
        "team": "home_team",
        "rating": "home_elo"
    }).sort_values("date"),
    on="date",
    by="home_team",
    direction="backward",
    allow_exact_matches=False
)

full = pd.merge_asof(
    home_elo.sort_values("date"),
    elo_lookup.rename(columns={
        "team": "away_team",
        "rating": "away_elo"
    }).sort_values("date"),
    on="date",
    by="away_team",
    direction="backward",
    allow_exact_matches=False
)


In [91]:
full.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo,away_elo
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,NaN,NaN
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,2003.0,1997.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1986.0,2014.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2006.0,1994.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1997.0,2003.0


In [92]:
full[["home_elo", "away_elo"]] = full[["home_elo", "away_elo"]].fillna(1500)

In [93]:
full.info()

<class 'pandas.DataFrame'>
RangeIndex: 49405 entries, 0 to 49404
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        49405 non-null  datetime64[us]
 1   home_team   49405 non-null  str           
 2   away_team   49405 non-null  str           
 3   home_score  49405 non-null  float64       
 4   away_score  49405 non-null  float64       
 5   tournament  49405 non-null  str           
 6   city        49405 non-null  str           
 7   country     49405 non-null  str           
 8   neutral     49405 non-null  bool          
 9   home_elo    49405 non-null  float64       
 10  away_elo    49405 non-null  float64       
dtypes: bool(1), datetime64[us](1), float64(4), str(5)
memory usage: 3.8 MB


In [94]:
full.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo,away_elo
49400,2026-06-09,Equatorial Guinea,Comoros,0.0,1.0,Friendly,Marrakesh,Morocco,True,1500.0,1359.0
49401,2026-06-10,Bolivia,Algeria,0.0,4.0,Friendly,Kansas City,United States,True,1675.0,1726.0
49402,2026-06-10,England,Costa Rica,3.0,0.0,Friendly,Orlando,United States,True,2042.0,1500.0
49403,2026-06-10,Portugal,Nigeria,2.0,1.0,Friendly,Leiria,Portugal,False,1976.0,1627.0
49404,2026-06-10,Afghanistan,Pakistan,0.0,2.0,Diamond Jubilee International Football Tournament,Malé,Maldives,True,1033.0,828.0


In [95]:
full.isna().sum()

date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
home_elo      0
away_elo      0
dtype: int64

In [96]:
 
df = full.reset_index(drop=True)
df["match_id"] = df.index


In [97]:
df.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo,away_elo,match_id
49400,2026-06-09,Equatorial Guinea,Comoros,0.0,1.0,Friendly,Marrakesh,Morocco,True,1500.0,1359.0,49400
49401,2026-06-10,Bolivia,Algeria,0.0,4.0,Friendly,Kansas City,United States,True,1675.0,1726.0,49401
49402,2026-06-10,England,Costa Rica,3.0,0.0,Friendly,Orlando,United States,True,2042.0,1500.0,49402
49403,2026-06-10,Portugal,Nigeria,2.0,1.0,Friendly,Leiria,Portugal,False,1976.0,1627.0,49403
49404,2026-06-10,Afghanistan,Pakistan,0.0,2.0,Diamond Jubilee International Football Tournament,Malé,Maldives,True,1033.0,828.0,49404


In [98]:
 
df = df.drop(columns=["change_x", "change_y"], errors="ignore")


In [99]:
df["elo_gap"] = df["home_elo"] - df["away_elo"]

In [100]:
df.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo,away_elo,match_id,elo_gap
49400,2026-06-09,Equatorial Guinea,Comoros,0.0,1.0,Friendly,Marrakesh,Morocco,True,1500.0,1359.0,49400,141.0
49401,2026-06-10,Bolivia,Algeria,0.0,4.0,Friendly,Kansas City,United States,True,1675.0,1726.0,49401,-51.0
49402,2026-06-10,England,Costa Rica,3.0,0.0,Friendly,Orlando,United States,True,2042.0,1500.0,49402,542.0
49403,2026-06-10,Portugal,Nigeria,2.0,1.0,Friendly,Leiria,Portugal,False,1976.0,1627.0,49403,349.0
49404,2026-06-10,Afghanistan,Pakistan,0.0,2.0,Diamond Jubilee International Football Tournament,Malé,Maldives,True,1033.0,828.0,49404,205.0


In [101]:
df = df.rename(columns={"home_team" : "team_1", "home_elo" : "team_1_elo", "away_team" : "team_2", "away_elo" : "team_2_elo", "home_score": "team_1_score", "away_score" : "team_2_score"})

In [102]:
df.head()

,date,team_1,team_2,team_1_score,team_2_score,tournament,city,country,neutral,team_1_elo,team_2_elo,match_id,elo_gap
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.0,1500.0,0,0.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,2003.0,1997.0,1,6.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1986.0,2014.0,2,-28.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2006.0,1994.0,3,12.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1997.0,2003.0,4,-6.0


In [103]:
df["neutral_encoded"]  = df["neutral"].astype("category").cat.codes

In [104]:
df.head()

,date,team_1,team_2,team_1_score,team_2_score,tournament,city,country,neutral,team_1_elo,team_2_elo,match_id,elo_gap,neutral_encoded
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.0,1500.0,0,0.0,0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,2003.0,1997.0,1,6.0,0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1986.0,2014.0,2,-28.0,0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2006.0,1994.0,3,12.0,0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1997.0,2003.0,4,-6.0,0


In [105]:
df["team_1_host"] = 1 - df["neutral_encoded"]


In [106]:
df.head()

,date,team_1,team_2,team_1_score,team_2_score,tournament,city,country,neutral,team_1_elo,team_2_elo,match_id,elo_gap,neutral_encoded,team_1_host
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.0,1500.0,0,0.0,0,1
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,2003.0,1997.0,1,6.0,0,1
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1986.0,2014.0,2,-28.0,0,1
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2006.0,1994.0,3,12.0,0,1
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1997.0,2003.0,4,-6.0,0,1


In [107]:
def match_result(row):
    if row["team_1_score"] > row["team_2_score"]:
        return "team 1 winner"
    elif row["team_1_score"] < row["team_2_score"]:
        return "team 2 winner"
    else:
        return "draw"

In [108]:
df["match_outcome"] = df.apply(match_result, axis=1)

In [109]:
df.head()

,date,team_1,team_2,team_1_score,team_2_score,tournament,city,country,neutral,team_1_elo,team_2_elo,match_id,elo_gap,neutral_encoded,team_1_host,match_outcome
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.0,1500.0,0,0.0,0,1,draw
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,2003.0,1997.0,1,6.0,0,1,team 1 winner
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1986.0,2014.0,2,-28.0,0,1,team 1 winner
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2006.0,1994.0,3,12.0,0,1,draw
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1997.0,2003.0,4,-6.0,0,1,team 1 winner


In [110]:
df["match_outcome_encoded"] = df["match_outcome"].astype("category").cat.codes

In [111]:
df.head()

,date,team_1,team_2,team_1_score,team_2_score,tournament,city,country,neutral,team_1_elo,team_2_elo,match_id,elo_gap,neutral_encoded,team_1_host,match_outcome,match_outcome_encoded
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.0,1500.0,0,0.0,0,1,draw,0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,2003.0,1997.0,1,6.0,0,1,team 1 winner,1
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1986.0,2014.0,2,-28.0,0,1,team 1 winner,1
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2006.0,1994.0,3,12.0,0,1,draw,0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1997.0,2003.0,4,-6.0,0,1,team 1 winner,1


In [112]:
df.corr(numeric_only=True)

,team_1_score,team_2_score,neutral,team_1_elo,team_2_elo,match_id,elo_gap,neutral_encoded,team_1_host,match_outcome_encoded
team_1_score,1.000000,-0.145296,-0.029239,0.130483,-0.196117,-0.098959,0.300594,-0.029239,0.029239,-0.114391
team_2_score,-0.145296,1.000000,0.080813,-0.151916,0.142330,-0.081571,-0.270291,0.080813,-0.080813,0.459156
neutral,-0.029239,0.080813,1.000000,-0.078549,-0.116838,0.076699,0.036674,1.000000,-1.000000,0.046307
team_1_elo,0.130483,-0.151916,-0.078549,1.000000,0.407865,-0.124694,0.533322,-0.078549,0.078549,-0.111927
team_2_elo,-0.196117,0.142330,-0.116838,0.407865,1.000000,-0.158740,-0.554830,-0.116838,0.116838,0.072489
match_id,-0.098959,-0.081571,0.076699,-0.124694,-0.158740,1.000000,0.033448,0.076699,-0.076699,-0.015743
elo_gap,0.300594,-0.270291,0.036674,0.533322,-0.554830,0.033448,1.000000,0.036674,-0.036674,-0.169147
neutral_encoded,-0.029239,0.080813,1.000000,-0.078549,-0.116838,0.076699,0.036674,1.000000,-1.000000,0.046307
team_1_host,0.029239,-0.080813,-1.000000,0.078549,0.116838,-0.076699,-0.036674,-1.000000,1.000000,-0.046307
match_outcome_encoded,-0.114391,0.459156,0.046307,-0.111927,0.072489,-0.015743,-0.169147,0.046307,-0.046307,1.000000


In [115]:
# Build rows from team_1's perspective.
team_1_rows = df[[
    "match_id",
    "date",
    "team_1",
    "team_2",
    "team_1_score",
    "team_2_score"
]].copy()

team_1_rows = team_1_rows.rename(columns={
    "team_1": "team",
    "team_2": "opponent",
    "team_1_score": "goals_for",
    "team_2_score": "goals_against"
})

# Build rows from team_2's perspective.
team_2_rows = df[[
    "match_id",
    "date",
    "team_2",
    "team_1",
    "team_2_score",
    "team_1_score"
]].copy()

team_2_rows = team_2_rows.rename(columns={
    "team_2": "team",
    "team_1": "opponent",
    "team_2_score": "goals_for",
    "team_1_score": "goals_against"
})

team_1_rows["date"] = pd.to_datetime(team_1_rows["date"])
team_2_rows["date"] = pd.to_datetime(team_2_rows["date"])
 


In [116]:
team_2_rows

,match_id,date,team,opponent,goals_for,goals_against
0,0,1872-11-30,England,Scotland,0.0,0.0
1,1,1873-03-08,Scotland,England,2.0,4.0
2,2,1874-03-07,England,Scotland,1.0,2.0
3,3,1875-03-06,Scotland,England,2.0,2.0
4,4,1876-03-04,England,Scotland,0.0,3.0
...,...,...,...,...,...,...
49400,49400,2026-06-09,Comoros,Equatorial Guinea,1.0,0.0
49401,49401,2026-06-10,Algeria,Bolivia,4.0,0.0
49402,49402,2026-06-10,Costa Rica,England,0.0,3.0
49403,49403,2026-06-10,Nigeria,Portugal,1.0,2.0


In [117]:
team_history = pd.concat([team_1_rows, team_2_rows], ignore_index=True)


team_history

,match_id,date,team,opponent,goals_for,goals_against
0,0,1872-11-30,Scotland,England,0.0,0.0
1,1,1873-03-08,England,Scotland,4.0,2.0
2,2,1874-03-07,Scotland,England,2.0,1.0
3,3,1875-03-06,England,Scotland,2.0,2.0
4,4,1876-03-04,Scotland,England,3.0,0.0
...,...,...,...,...,...,...
98805,49400,2026-06-09,Comoros,Equatorial Guinea,1.0,0.0
98806,49401,2026-06-10,Algeria,Bolivia,4.0,0.0
98807,49402,2026-06-10,Costa Rica,England,0.0,3.0
98808,49403,2026-06-10,Nigeria,Portugal,1.0,2.0


In [118]:
# Goal difference from each team's perspective.
team_history["goal_difference"] =  team_history["goals_for"] - team_history["goals_against"]
team_history.head()



,match_id,date,team,opponent,goals_for,goals_against,goal_difference
0,0,1872-11-30,Scotland,England,0.0,0.0,0.0
1,1,1873-03-08,England,Scotland,4.0,2.0,2.0
2,2,1874-03-07,Scotland,England,2.0,1.0,1.0
3,3,1875-03-06,England,Scotland,2.0,2.0,0.0
4,4,1876-03-04,Scotland,England,3.0,0.0,3.0


In [119]:
# Create win/draw/loss flags.
# True/False becomes 1/0 using astype(int).
team_history["win"] = (
    team_history["goals_for"] > team_history["goals_against"]
).astype(int)

team_history["draw"] = (
    team_history["goals_for"] == team_history["goals_against"]
).astype(int)

team_history["loss"] = (
    team_history["goals_for"] < team_history["goals_against"]
).astype(int)



In [120]:
def calculate_points(row):
    if row["win"] == 1:
        return 3
    elif row["draw"] == 1:
        return 1
    else:
        return 0

team_history["form_points"] = team_history.apply(calculate_points, axis=1)

In [121]:
# Sort by team first, then date.
# This makes each team's matches chronological inside their group.
team_history = team_history.sort_values(["team", "date"]).reset_index(drop=True)
#first sort out by teams alphabetcally, then by the date


In [122]:
team_history.groupby("team")

In [123]:
england_rows = team_history[team_history["team"] == "England"]
england_rows.head(20)

,match_id,date,team,opponent,goals_for,goals_against,goal_difference,win,draw,loss,form_points
24842,0,1872-11-30,England,Scotland,0.0,0.0,0.0,0,1,0,1
24843,1,1873-03-08,England,Scotland,4.0,2.0,2.0,1,0,0,3
24844,2,1874-03-07,England,Scotland,1.0,2.0,-1.0,0,0,1,0
24845,3,1875-03-06,England,Scotland,2.0,2.0,0.0,0,1,0,1
24846,4,1876-03-04,England,Scotland,0.0,3.0,-3.0,0,0,1,0
24847,6,1877-03-03,England,Scotland,1.0,3.0,-2.0,0,0,1,0
24848,8,1878-03-02,England,Scotland,2.0,7.0,-5.0,0,0,1,0
24849,10,1879-01-18,England,Wales,2.0,1.0,1.0,1,0,0,3
24850,11,1879-04-05,England,Scotland,5.0,4.0,1.0,1,0,0,3
24851,13,1880-03-13,England,Scotland,4.0,5.0,-1.0,0,0,1,0


In [124]:

# Average goals scored in previous 5 games.
team_history["last5_goals_for_avg"] = (
    team_history
    .groupby("team")["goals_for"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

# Average goals conceded in previous 5 games.
team_history["last5_goals_against_avg"] = (
    team_history
    .groupby("team")["goals_against"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

# Average goal difference in previous 5 games.
team_history["last5_goal_difference_avg"] = (
    team_history
    .groupby("team")["goal_difference"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

# Number of wins in previous 5 games.
team_history["last5_wins"] = (
    team_history
    .groupby("team")["win"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)

# Number of draws in previous 5 games.
team_history["last5_draws"] = (
    team_history
    .groupby("team")["draw"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)

# Number of losses in previous 5 games.
team_history["last5_losses"] = (
    team_history
    .groupby("team")["loss"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)

# Total form points in previous 5 games.
team_history["last5_form_points"] = (
    team_history
    .groupby("team")["form_points"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)

In [125]:
team_history.head()

,match_id,date,team,opponent,goals_for,goals_against,goal_difference,win,draw,loss,form_points,last5_goals_for_avg,last5_goals_against_avg,last5_goal_difference_avg,last5_wins,last5_draws,last5_losses,last5_form_points
0,36260,2012-09-25,Abkhazia,Artsakh,1.0,1.0,0.0,0,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,36404,2012-10-21,Abkhazia,Artsakh,0.0,3.0,-3.0,0,0,1,0,1.000000,1.000000,0.0,0.0,1.0,0.0,1.0
2,37317,2013-09-23,Abkhazia,South Ossetia,3.0,0.0,3.0,1,0,0,3,0.500000,2.000000,-1.5,0.0,1.0,1.0,1.0
3,37779,2014-06-01,Abkhazia,Occitania,1.0,1.0,0.0,0,1,0,1,1.333333,1.333333,0.0,1.0,1.0,1.0,4.0
4,37794,2014-06-02,Abkhazia,Sápmi,2.0,1.0,1.0,1,0,0,3,1.250000,1.250000,0.0,1.0,2.0,1.0,5.0


In [126]:
eng = team_history[team_history["team"]=="England"]
eng

,match_id,date,team,opponent,goals_for,goals_against,goal_difference,win,draw,loss,form_points,last5_goals_for_avg,last5_goals_against_avg,last5_goal_difference_avg,last5_wins,last5_draws,last5_losses,last5_form_points
24842,0,1872-11-30,England,Scotland,0.0,0.0,0.0,0,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24843,1,1873-03-08,England,Scotland,4.0,2.0,2.0,1,0,0,3,0.000000,0.000000,0.000000,0.0,1.0,0.0,1.0
24844,2,1874-03-07,England,Scotland,1.0,2.0,-1.0,0,0,1,0,2.000000,1.000000,1.000000,1.0,1.0,0.0,4.0
24845,3,1875-03-06,England,Scotland,2.0,2.0,0.0,0,1,0,1,1.666667,1.333333,0.333333,1.0,1.0,1.0,4.0
24846,4,1876-03-04,England,Scotland,0.0,3.0,-3.0,0,0,1,0,1.750000,1.500000,0.250000,1.0,2.0,1.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25927,48943,2025-11-16,England,Albania,2.0,0.0,2.0,1,0,0,3,3.400000,0.000000,3.400000,5.0,0.0,0.0,15.0
25928,49168,2026-03-27,England,Uruguay,1.0,1.0,0.0,0,1,0,1,3.400000,0.000000,3.400000,5.0,0.0,0.0,15.0
25929,49247,2026-03-31,England,Japan,0.0,1.0,-1.0,0,0,1,0,2.600000,0.200000,2.400000,4.0,1.0,0.0,13.0
25930,49364,2026-06-06,England,New Zealand,1.0,0.0,1.0,1,0,0,3,2.000000,0.400000,1.600000,3.0,1.0,1.0,10.0


In [127]:
# Choose only the rolling feature columns we want to merge back.
# match_id makes this merge one-to-one even if a team has multiple matches on the same date.
rolling_features = team_history[[
    "match_id",
    "team",
    "last5_goals_for_avg",
    "last5_goals_against_avg",
    "last5_goal_difference_avg",
    "last5_wins",
    "last5_draws",
    "last5_losses",
    "last5_form_points"
]].copy()

# Merge team_1's rolling features.
df = df.merge(
    rolling_features.rename(columns={
        "team": "team_1",
        "last5_goals_for_avg": "team_1_last5_goals_for_avg",
        "last5_goals_against_avg": "team_1_last5_goals_against_avg",
        "last5_goal_difference_avg": "team_1_last5_goal_difference_avg",
        "last5_wins": "team_1_last5_wins",
        "last5_draws": "team_1_last5_draws",
        "last5_losses": "team_1_last5_losses",
        "last5_form_points": "team_1_last5_form_points"
    }),
    on=["match_id", "team_1"],
    how="left"
)

# Merge team_2's rolling features.
df = df.merge(
    rolling_features.rename(columns={
        "team": "team_2",
        "last5_goals_for_avg": "team_2_last5_goals_for_avg",
        "last5_goals_against_avg": "team_2_last5_goals_against_avg",
        "last5_goal_difference_avg": "team_2_last5_goal_difference_avg",
        "last5_wins": "team_2_last5_wins",
        "last5_draws": "team_2_last5_draws",
        "last5_losses": "team_2_last5_losses",
        "last5_form_points": "team_2_last5_form_points"
    }),
    on=["match_id", "team_2"],
    how="left"
)


In [128]:
# Difference features are useful because the model often cares about
# team_1 relative strength compared with team_2.
df["last5_goals_for_avg_diff"] = (
    df["team_1_last5_goals_for_avg"] - df["team_2_last5_goals_for_avg"]
)

df["last5_goals_against_avg_diff"] = (
    df["team_1_last5_goals_against_avg"] - df["team_2_last5_goals_against_avg"]
)

df["last5_goal_difference_avg_diff"] = (
    df["team_1_last5_goal_difference_avg"] - df["team_2_last5_goal_difference_avg"]
)

df["last5_form_points_diff"] = (
    df["team_1_last5_form_points"] - df["team_2_last5_form_points"]
)

In [129]:
df.tail()

,date,team_1,team_2,team_1_score,team_2_score,tournament,city,country,neutral,team_1_elo,...,team_2_last5_goals_against_avg,team_2_last5_goal_difference_avg,team_2_last5_wins,team_2_last5_draws,team_2_last5_losses,team_2_last5_form_points,last5_goals_for_avg_diff,last5_goals_against_avg_diff,last5_goal_difference_avg_diff,last5_form_points_diff
49400,2026-06-09,Equatorial Guinea,Comoros,0.0,1.0,Friendly,Marrakesh,Morocco,True,1500.0,...,0.6,-0.6,0.0,3.0,2.0,3.0,0.8,0.8,0.0,1.0
49401,2026-06-10,Bolivia,Algeria,0.0,4.0,Friendly,Kansas City,United States,True,1675.0,...,0.4,1.4,3.0,1.0,1.0,10.0,-0.6,1.2,-1.8,-4.0
49402,2026-06-10,England,Costa Rica,3.0,0.0,Friendly,Orlando,United States,True,2042.0,...,2.2,-1.6,0.0,2.0,3.0,2.0,0.6,-1.8,2.4,8.0
49403,2026-06-10,Portugal,Nigeria,2.0,1.0,Friendly,Leiria,Portugal,False,1976.0,...,1.0,1.2,3.0,2.0,0.0,11.0,0.4,-0.2,0.6,-1.0
49404,2026-06-10,Afghanistan,Pakistan,0.0,2.0,Diamond Jubilee International Football Tournament,Malé,Maldives,True,1033.0,...,1.2,0.0,2.0,2.0,1.0,8.0,-0.8,-0.4,-0.4,-3.0


In [130]:
df.isna().sum()

date                                  0
team_1                                0
team_2                                0
team_1_score                          0
team_2_score                          0
tournament                            0
city                                  0
country                               0
neutral                               0
team_1_elo                            0
team_2_elo                            0
match_id                              0
elo_gap                               0
neutral_encoded                       0
team_1_host                           0
match_outcome                         0
match_outcome_encoded                 0
team_1_last5_goals_for_avg          141
team_1_last5_goals_against_avg      141
team_1_last5_goal_difference_avg    141
team_1_last5_wins                   141
team_1_last5_draws                  141
team_1_last5_losses                 141
team_1_last5_form_points            141
team_2_last5_goals_for_avg          195


In [131]:

def get_tournament_category(tournament):
    competition = str(tournament).lower().strip()

    if competition == "friendly":
        return "friendly"

    if "qualification" in competition or "qualifier" in competition:
        return "qualifier"

    if  "fifa world cup" in competition:
        return "world_cup"

    if "nations league" in competition:
        return "nations_league"

    continental_keywords = [
        "uefa euro",
        "copa américa",
        "copa america",
        "african cup of nations",
        "afc asian cup",
        "concacaf championship",
        "gold cup",
        "oceania nations cup"
    ]

    if any(keyword in competition for keyword in continental_keywords):
        return "continental_tournament"
    else:
        return "other"


tournament_weights = {
    "world_cup": 55,
    "continental_tournament": 37.5,
    "qualifier": 25,
    "nations_league": 20,
    "friendly": 7.5,
    "other": 12.5
}

df["tournament_category"] = df["tournament"].apply(get_tournament_category)
df["tournament_weight"] = df["tournament_category"].map(tournament_weights)
 

In [132]:
df.head()

,date,team_1,team_2,team_1_score,team_2_score,tournament,city,country,neutral,team_1_elo,...,team_2_last5_wins,team_2_last5_draws,team_2_last5_losses,team_2_last5_form_points,last5_goals_for_avg_diff,last5_goals_against_avg_diff,last5_goal_difference_avg_diff,last5_form_points_diff,tournament_category,tournament_weight
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,friendly,7.5
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,2003.0,...,0.0,1.0,0.0,1.0,0.000000,0.000000,0.000000,0.0,friendly,7.5
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1986.0,...,1.0,1.0,0.0,4.0,-1.000000,1.000000,-2.000000,-3.0,friendly,7.5
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2006.0,...,1.0,1.0,1.0,4.0,0.333333,-0.333333,0.666667,0.0,friendly,7.5
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1997.0,...,1.0,2.0,1.0,5.0,-0.250000,0.250000,-0.500000,0.0,friendly,7.5


In [133]:

df["date"] = pd.to_datetime(df["date"])


latest_date = df["date"].max()


df["years_ago"] = (latest_date - df["date"]).dt.days / 365


def match_recency(years_ago):
    if years_ago <= 1:
        return "0_to_12_months"
    elif years_ago <= 2:
        return "12_to_24_months"
    elif years_ago <= 3:
        return "24_to_36_months"
    elif years_ago <= 4:
        return "36_to_48_months"
    else:
        return "older_than_48_months"

df["match_recency"] = df["years_ago"].apply(match_recency)

 
match_recency_weights = {
    "0_to_12_months": 1.0,
    "12_to_24_months": 0.5,
    "24_to_36_months": 0.3,
    "36_to_48_months": 0.2,
    "older_than_48_months": 0.0
}

df["recency_weight"] = df["match_recency"].map(match_recency_weights)


df["match_weighting"] = df["tournament_weight"] * df["recency_weight"]
 

In [134]:
df.head()

,date,team_1,team_2,team_1_score,team_2_score,tournament,city,country,neutral,team_1_elo,...,last5_goals_for_avg_diff,last5_goals_against_avg_diff,last5_goal_difference_avg_diff,last5_form_points_diff,tournament_category,tournament_weight,years_ago,match_recency,recency_weight,match_weighting
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.0,...,NaN,NaN,NaN,NaN,friendly,7.5,153.627397,older_than_48_months,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,2003.0,...,0.000000,0.000000,0.000000,0.0,friendly,7.5,153.358904,older_than_48_months,0.0,0.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1986.0,...,-1.000000,1.000000,-2.000000,-3.0,friendly,7.5,152.361644,older_than_48_months,0.0,0.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2006.0,...,0.333333,-0.333333,0.666667,0.0,friendly,7.5,151.364384,older_than_48_months,0.0,0.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1997.0,...,-0.250000,0.250000,-0.500000,0.0,friendly,7.5,150.367123,older_than_48_months,0.0,0.0


In [135]:
df.columns

Index(['date', 'team_1', 'team_2', 'team_1_score', 'team_2_score',
       'tournament', 'city', 'country', 'neutral', 'team_1_elo', 'team_2_elo',
       'match_id', 'elo_gap', 'neutral_encoded', 'team_1_host',
       'match_outcome', 'match_outcome_encoded', 'team_1_last5_goals_for_avg',
       'team_1_last5_goals_against_avg', 'team_1_last5_goal_difference_avg',
       'team_1_last5_wins', 'team_1_last5_draws', 'team_1_last5_losses',
       'team_1_last5_form_points', 'team_2_last5_goals_for_avg',
       'team_2_last5_goals_against_avg', 'team_2_last5_goal_difference_avg',
       'team_2_last5_wins', 'team_2_last5_draws', 'team_2_last5_losses',
       'team_2_last5_form_points', 'last5_goals_for_avg_diff',
       'last5_goals_against_avg_diff', 'last5_goal_difference_avg_diff',
       'last5_form_points_diff', 'tournament_category', 'tournament_weight',
       'years_ago', 'match_recency', 'recency_weight', 'match_weighting'],
      dtype='str')

In [136]:
df.head()

,date,team_1,team_2,team_1_score,team_2_score,tournament,city,country,neutral,team_1_elo,...,last5_goals_for_avg_diff,last5_goals_against_avg_diff,last5_goal_difference_avg_diff,last5_form_points_diff,tournament_category,tournament_weight,years_ago,match_recency,recency_weight,match_weighting
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.0,...,NaN,NaN,NaN,NaN,friendly,7.5,153.627397,older_than_48_months,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,2003.0,...,0.000000,0.000000,0.000000,0.0,friendly,7.5,153.358904,older_than_48_months,0.0,0.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1986.0,...,-1.000000,1.000000,-2.000000,-3.0,friendly,7.5,152.361644,older_than_48_months,0.0,0.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2006.0,...,0.333333,-0.333333,0.666667,0.0,friendly,7.5,151.364384,older_than_48_months,0.0,0.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1997.0,...,-0.250000,0.250000,-0.500000,0.0,friendly,7.5,150.367123,older_than_48_months,0.0,0.0


In [137]:
df.isna().sum()

date                                  0
team_1                                0
team_2                                0
team_1_score                          0
team_2_score                          0
tournament                            0
city                                  0
country                               0
neutral                               0
team_1_elo                            0
team_2_elo                            0
match_id                              0
elo_gap                               0
neutral_encoded                       0
team_1_host                           0
match_outcome                         0
match_outcome_encoded                 0
team_1_last5_goals_for_avg          141
team_1_last5_goals_against_avg      141
team_1_last5_goal_difference_avg    141
team_1_last5_wins                   141
team_1_last5_draws                  141
team_1_last5_losses                 141
team_1_last5_form_points            141
team_2_last5_goals_for_avg          195


In [138]:
model_df = df[df["match_weighting"]>0.0]

model_df.to_csv("model_dataset.csv", index=False)

In [139]:
model_df["team_1_elo"] = model_df["team_1_elo"].fillna(1500)
model_df["team_2_elo"] = model_df["team_2_elo"].fillna(1500)

model_df["elo_gap"] = model_df["team_1_elo"] - model_df["team_2_elo"]

In [140]:
model_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4057 entries, 45348 to 49404
Data columns (total 41 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   date                              4057 non-null   datetime64[us]
 1   team_1                            4057 non-null   str           
 2   team_2                            4057 non-null   str           
 3   team_1_score                      4057 non-null   float64       
 4   team_2_score                      4057 non-null   float64       
 5   tournament                        4057 non-null   str           
 6   city                              4057 non-null   str           
 7   country                           4057 non-null   str           
 8   neutral                           4057 non-null   bool          
 9   team_1_elo                        4057 non-null   float64       
 10  team_2_elo                        4057 non-null   floa

In [141]:
model_df.tail()

,date,team_1,team_2,team_1_score,team_2_score,tournament,city,country,neutral,team_1_elo,...,last5_goals_for_avg_diff,last5_goals_against_avg_diff,last5_goal_difference_avg_diff,last5_form_points_diff,tournament_category,tournament_weight,years_ago,match_recency,recency_weight,match_weighting
49400,2026-06-09,Equatorial Guinea,Comoros,0.0,1.0,Friendly,Marrakesh,Morocco,True,1500.0,...,0.8,0.8,0.0,1.0,friendly,7.5,0.00274,0_to_12_months,1.0,7.5
49401,2026-06-10,Bolivia,Algeria,0.0,4.0,Friendly,Kansas City,United States,True,1675.0,...,-0.6,1.2,-1.8,-4.0,friendly,7.5,0.00000,0_to_12_months,1.0,7.5
49402,2026-06-10,England,Costa Rica,3.0,0.0,Friendly,Orlando,United States,True,2042.0,...,0.6,-1.8,2.4,8.0,friendly,7.5,0.00000,0_to_12_months,1.0,7.5
49403,2026-06-10,Portugal,Nigeria,2.0,1.0,Friendly,Leiria,Portugal,False,1976.0,...,0.4,-0.2,0.6,-1.0,friendly,7.5,0.00000,0_to_12_months,1.0,7.5
49404,2026-06-10,Afghanistan,Pakistan,0.0,2.0,Diamond Jubilee International Football Tournament,Malé,Maldives,True,1033.0,...,-0.8,-0.4,-0.4,-3.0,other,12.5,0.00000,0_to_12_months,1.0,12.5


In [142]:
rolling_cols = [
    "team_1_last5_goals_for_avg",
    "team_1_last5_goals_against_avg",
    "team_1_last5_goal_difference_avg",
    "team_1_last5_wins",
    "team_1_last5_draws",
    "team_1_last5_losses",
    "team_1_last5_form_points",
    "team_2_last5_goals_for_avg",
    "team_2_last5_goals_against_avg",
    "team_2_last5_goal_difference_avg",
    "team_2_last5_wins",
    "team_2_last5_draws",
    "team_2_last5_losses",
    "team_2_last5_form_points",
     "last5_goals_for_avg_diff",              
   "last5_goals_against_avg_diff",          
   "last5_goal_difference_avg_diff",       
   "last5_form_points_diff"
]

model_df[rolling_cols] = model_df[rolling_cols].fillna(0)

In [143]:
model_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4057 entries, 45348 to 49404
Data columns (total 41 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   date                              4057 non-null   datetime64[us]
 1   team_1                            4057 non-null   str           
 2   team_2                            4057 non-null   str           
 3   team_1_score                      4057 non-null   float64       
 4   team_2_score                      4057 non-null   float64       
 5   tournament                        4057 non-null   str           
 6   city                              4057 non-null   str           
 7   country                           4057 non-null   str           
 8   neutral                           4057 non-null   bool          
 9   team_1_elo                        4057 non-null   float64       
 10  team_2_elo                        4057 non-null   floa

In [144]:
model_df = model_df.to_csv("model_dataset.csv",index=False)

In [ ]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss



model_df = pd.read_csv("model_dataset.csv")

model_df["date"] = pd.to_datetime(model_df["date"])
model_df = model_df.sort_values("date").reset_index(drop=True)





features = [
    "team_1_elo",
    "team_2_elo",
    "elo_gap",

    "neutral_encoded",
    "team_1_host",

    "team_1_last5_goals_for_avg",
    "team_1_last5_goals_against_avg",
    "team_1_last5_goal_difference_avg",
    "team_1_last5_wins",
    "team_1_last5_draws",
    "team_1_last5_losses",
    "team_1_last5_form_points",

    "team_2_last5_goals_for_avg",
    "team_2_last5_goals_against_avg",
    "team_2_last5_goal_difference_avg",
    "team_2_last5_wins",
    "team_2_last5_draws",
    "team_2_last5_losses",
    "team_2_last5_form_points",
    "tournament_weight"
  
]
target = "match_outcome_encoded"

 

 
X = model_df[features]
y = model_df[target]
sample_weight = model_df["match_weighting"]

 

split_index = int(len(model_df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

sample_weight_train = sample_weight.iloc[:split_index]
sample_weight_test = sample_weight.iloc[split_index:]

 

 
model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=400,
    learning_rate=0.03,
    max_depth=3,
    eval_metric="mlogloss",
     
)

 
model.fit(
    X_train,
    y_train,
    sample_weight=sample_weight_train
)


 
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)


 
accuracy = accuracy_score(y_test, y_pred)
loss = log_loss(y_test, y_pred_proba)

print("Model Accuracy:", accuracy)
print("Model Log Loss:", loss, '\n')

 
class_names = ["Draw", "Team 1 Win", "Team 2 Win"]
print(classification_report(
    y_test,
    y_pred,
     target_names=class_names
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


 
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

 
y_train_pred = model.predict(X_train)
y_train_proba = model.predict_proba(X_train)
 
accuracy = accuracy_score(y_test, y_pred)
loss = log_loss(y_test, y_pred_proba)
 
accuracy_train = accuracy_score(y_train, y_train_pred)
loss_train = log_loss(y_train, y_train_proba)


 


Model Accuracy: 0.6022167487684729
Model Log Loss: 0.8966873560958508
              precision    recall  f1-score   support

        Draw       0.36      0.11      0.17       186
  Team 1 Win       0.64      0.85      0.73       390
  Team 2 Win       0.58      0.58      0.58       236

    accuracy                           0.60       812
   macro avg       0.53      0.51      0.49       812
weighted avg       0.56      0.60      0.56       812


Confusion Matrix:
[[ 21 111  54]
 [ 14 331  45]
 [ 24  75 137]]


In [146]:

print("Training Set Accuracy:", accuracy_train)
print("Training Set Log Loss:", loss_train, '\n\n')

print("Test Set Accuracy:", accuracy)
print("Test Set Log Loss:", loss)


Training Set Accuracy: 0.637904468412943
Training Set Log Loss: 0.8044653557570293 


Test Set Accuracy: 0.6022167487684729
Test Set Log Loss: 0.8966873560958508
